# Cửa sổ rerank: quét k × trọng số CE — chỉ DEV

Chẩn đoán 2026-09-20: cửa sổ rerank top-20 đang là trần của pipeline luồng mô tả tự do.

| độ sâu | R@10 | R@20 | R@50 | R@100 |
|---|---:|---:|---:|---:|
| fusion (paraphrase dev, có dấu) | 0,702 | 0,758 | 0,824 | 0,864 |

Pipeline đạt **0,718**, tức gần hết trần của cửa sổ 20 (0,758). Nới cửa sổ nâng trần lên
0,824 (k=50) hoặc 0,864 (k=100).

Công thức blend chuẩn hoá hạng hybrid **theo kích thước cửa sổ**, nên một tài liệu chỉ
vào được top-10 khi hạng hybrid < `10 + (w/(1-w))·(k-1)`. Với w = 0,3 thì k = 100 cũng
chỉ với tới hạng 52. Vì vậy phải quét **k và trọng số cùng lúc**.

Đã chạy trên CPU bằng cache cũ (k ≤ 30): `k=30, w=0.5` cho +0,016 R@10 trên cả hai lát
paraphrase nhưng title không dấu −0,0101, **trượt** quy tắc an toàn. Notebook này đo phần
còn thiếu: k = 50 và 100.

**Quy tắc chốt, đặt trước khi chạy** (script tự áp dụng):
- Chỉ đổi khi delta R@10 dương và CI không chứa 0 trên **cả hai** lát paraphrase.
- Không lát nào giảm quá 0,01 R@10 hoặc MRR@10.
- Nhiều cấu hình đạt thì lấy k nhỏ nhất, vì k lớn tốn thời gian rerank.
- DEV only. Xác nhận trên test là một lần chạy riêng, sau khi đã chốt.

**Trước khi chạy:** Runtime → Change runtime type → T4 GPU. Dữ liệu Jira đã scrub nhưng
vẫn chứa tên service, mã lỗi và luồng nghiệp vụ; kiểm tra quy định đơn vị trước khi upload.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "sentence-transformers==6.0.1" gdown

## 1. Tải bundle lên và giải nén

In [ ]:
from google.colab import files
up = files.upload()  # chọn colab/rerank_window_bundle.zip
BUNDLE = next(iter(up))
!unzip -oq {BUNDLE} -d /content/work
%cd /content/work
!ls pipeline data/processed

## 2. Tải đúng hai checkpoint pilot

Cùng hai file Drive đã dùng cho `rerank_k_colab.ipynb`.

In [ ]:
import os, glob, shutil, zipfile, gdown
FILES = {"e5-base-task2": "1rNO17nLa1kXGdmMEsUhW5Hl_9e7kjN8O",
         "bge-reranker-base-task": "12x-H0yLkedkl0_94K_CHV621cZuncRqn"}
os.makedirs("models", exist_ok=True)
for name, fid in FILES.items():
    dst, z = "models/" + name, "/content/" + name + ".zip"
    if not os.path.exists(dst + "/config.json"):
        gdown.download(id=fid, output=z, quiet=False)
        assert zipfile.is_zipfile(z), "Không tải được model ZIP: " + name
        tmp = "/content/unz_" + name
        shutil.rmtree(tmp, ignore_errors=True)
        with zipfile.ZipFile(z) as archive: archive.extractall(tmp)
        cfg = sorted(glob.glob(tmp + "/**/config.json", recursive=True), key=len)
        shutil.move(os.path.dirname(cfg[0]), dst)
    print(name, os.path.exists(dst + "/config.json"))

## 3. Smoke test 2 query

Chỉ để chắc đường dẫn và model đúng. Kết quả số ở bước này không có ý nghĩa.

In [ ]:
!python pipeline/eval_rerank_window.py --device cuda --limit 2 --head 50     --ks 10 20 50 --weights 0.3 0.5 --n-boot 100     --cache /content/smoke_scores.jsonl --out /content/smoke.json

## 4. Chạy toàn bộ DEV

Chấm CE cho top-100 của 1.990 lượt (paraphrase + title, có dấu và không dấu), khoảng
199.000 cặp. Trên T4 mất chừng 10–15 phút. Mọi (k, w) sau đó được tính lại trên cùng bộ
điểm này nên thêm cấu hình không tốn thêm GPU.

Nếu phiên Colab đứt, chạy lại đúng ô này: điểm CE đã chấm nằm trong cache và được dùng lại.

In [ ]:
!python pipeline/eval_rerank_window.py --device cuda --head 100     --ks 10 20 30 50 100 --weights 0.3 0.4 0.5 0.6 0.8 1.0     --ce-batch 128 --score-chunk 50 --n-boot 5000

## 5. Tải kết quả về

Giải nén vào `D:/viettel/semantic/` (các file đã nằm sẵn đường dẫn `data/processed/`),
rồi báo lại để đối chiếu và quyết định có đổi cấu hình pilot hay không.

Giữ cả file cache: nó cho phép thử thêm (k, w) khác trên CPU mà không cần chấm CE lại.

In [ ]:
!zip -q rerank_window_results.zip data/processed/rerank_window_dev.json data/processed/rerank_window_dev_scores.jsonl
!ls -lh rerank_window_results.zip
files.download("rerank_window_results.zip")